In [1]:
import pandas as pd
import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

df = pd.read_csv("hasil_back_translation_redbus_final.csv", encoding="utf-8-sig")
aspects = ["Ticketing", "Information Channels", "Travel Experience"]

label_cols = []
for col in aspects:
    clean_series = df[col].astype(str).str.lower().str.strip()
    
    pos_col, neg_col = f"{col}_Positif", f"{col}_Negatif"
    df[pos_col] = (clean_series == 'positif').astype(np.int8)
    df[neg_col] = (clean_series == 'negatif').astype(np.int8)
    label_cols.extend([pos_col, neg_col])

X = np.zeros((len(df), 1)) 
y = df[label_cols].to_numpy(dtype=np.int8)

msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, temp_idx = next(msss1.split(X, y))

msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_rel_idx, test_rel_idx = next(msss2.split(X[temp_idx], y[temp_idx]))

val_idx, test_idx = temp_idx[val_rel_idx], temp_idx[test_rel_idx]

splits = {
    "train_data_encoded.csv": train_idx,
    "val_data_encoded.csv": val_idx,
    "test_data_encoded.csv": test_idx
}

print("Hasil Distribusi Multilabel Stratified Splitting:")
for filename, indices in splits.items():
    df.iloc[indices].to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"- {filename.split('_')[0].capitalize():<10}: {len(indices)}")

Hasil Distribusi Multilabel Stratified Splitting:
- Train     : 1573
- Val       : 201
- Test      : 203
